In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data, util

def load_image(path=''):
    if not path.strip():
        return data.camera()
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4), squeeze=False)
    for ax, im, title in zip(axes.ravel(), images, titles):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

img = load_image(input('Image path (Enter for sample): '))

In [ ]:
# 1
from skimage.metrics import peak_signal_noise_ratio
s = int(input('Initial odd window size: '))
smax = int(input('Maximum odd window size: '))
assert 3<=s<=smax and s%2==1 and smax%2==1
noisy = np.round(util.random_noise(img,mode='s&p',amount=.1)*255).astype(np.uint8)
def adaptive_median(im,s,smax):
    r=smax//2
    padded=np.pad(im,r,mode='edge')
    out=np.empty_like(im)
    for i in range(im.shape[0]):
        for j in range(im.shape[1]):
            z=im[i,j]
            for size in range(s,smax+1,2):
                t=size//2
                window=padded[i+r-t:i+r+t+1,j+r-t:j+r+t+1]
                low,med,high=window.min(),np.median(window),window.max()
                if low<med<high:
                    out[i,j]=z if low<z<high else med
                    break
            else:
                out[i,j]=med
    return out
median=cv2.medianBlur(noisy,s)
adaptive=adaptive_median(noisy,s,smax)
show([img,noisy,median,adaptive],['Original','Noisy','Median','Adaptive median'])
for name,out in [('Median',median),('Adaptive median',adaptive)]:
    print(name,'PSNR:',peak_signal_noise_ratio(img,out,data_range=255),'dB')

In [ ]:
# 2
hist=np.bincount(img.ravel(),minlength=256).astype(float)
p=hist/hist.sum()
w=np.cumsum(p)
m=np.cumsum(p*np.arange(256))
variance=np.divide((m[-1]*w-m)**2,w*(1-w),out=np.zeros(256),where=(w>0)&(w<1))
threshold=int(np.argmax(variance))
binary=np.where(img>threshold,255,0).astype(np.uint8)
print('Otsu threshold:',threshold)
show([img,binary],['Original','Otsu segmentation'])